### ⚖️ Section 2: Two Approaches to Guardrails
#### 1. Deterministic Guardrails
- Rule-based: regex, keyword matching, explicit checks
- ✅ Fast, predictable, cost-effective
- ❌ May miss nuanced violations

In [1]:


import re

def detetministric_g(text:str)->bool:
    harmful_words = ["hack","steal","virus","malware"]
    return any((kw in text.lower()) for kw in harmful_words)


text_lst =[
    "how to learn hacking in ai ?",
    "what is use of Ai",
    "how to create virus"
]

for i in text_lst:
    b= detetministric_g(i)
    status = "🚫 BLOCKED" if b else "✅ ALLOWED"
    print(f"{status} : {i}")
    

🚫 BLOCKED : how to learn hacking in ai ?
✅ ALLOWED : what is use of Ai
🚫 BLOCKED : how to create virus



#### Model-Based Guardrails
- Uses LLMs/classifiers for semantic understanding
- ✅ Catches subtle/nuanced issues
- ❌ Slower and more expensive<br>
- strip() : removes extra spaces and new lines from the beginning and end of text.   
- Use f""" when you want to insert variables:            

In [2]:


import os 
from langchain_openai import  ChatOpenAI
from langchain.messages import  HumanMessage

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

def model_base_guardrails(text:str)->str:
    """ check the correct input of not"""

    model = ChatOpenAI(model="gpt-4o-mini")
    prompt = f"""
    
    is the following user input is the safe of process ?
    reply only one word (safe or unsafe)

    input :{text}
    """

    ask = model.invoke([HumanMessage(content=prompt)])

    return ask.content.strip()



for i in text_lst:
    check = model_base_guardrails(i)
    status = "🚫 unsafe" if "unsafe"in check else "✅ safe"
    print(f"{status} : {i}")

    

🚫 unsafe : how to learn hacking in ai ?
✅ safe : what is use of Ai
🚫 unsafe : how to create virus



### Built-in Guardrail — PII Detection Middleware
LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [23]:

import os
from langchain.agents import  create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import  tool

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

@tool
def userinformation(query:str)->str:
    """check the user information :"""

    return f" i found the user query : {query}"

agent = create_agent(
    model='gpt-4o',
    tools=[userinformation],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input = True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "ip",
            strategy="hash",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)

In [34]:

respone = agent.invoke({"messages":[{"role":"user","content":" i am johncena@gmail.com and my credit_card is 5105-1051-0510-5100, i have ip adrres is 129.101.1.1"}]})
respone["messages"][0].content

' i am [REDACTED_EMAIL] and my credit_card is ****-****-****-5100, i have ip adrres is <ip_hash:1936e62e>'

In [36]:
try:
    respone = agent.invoke({"messages":[{"role":"user","content":"Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"}]})


except Exception as e:
    print(e)

Detected 1 instance(s) of api_key in text content


#### Built-in Guardrail — Human-in-the-Loop Middleware
Pauses agent execution before sensitive operations and waits for human approval.

Best for:

- Financial transactions
- Sending emails to external parties
= Deleting production data
- Any operation with significant business impact

In [4]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import  tool
from langgraph.checkpoint.memory import  InMemorySaver
from langgraph.types import Command

@tool
def search_web(query:str)->str:
    """ search the web for information """
    return f" seach the web by query :{query}"

@tool
def send_email(recipient:str,subject:str,body:str)->str:
    """send the email to recipeint """
    return f"send the Email to {recipient} with subject : {subject}"

@tool
def delete_record(table:str,condition:str)->str:
    """delete the record from the database"""
    return f"delete the record from the table {table} where condition : {condition}"


agent = create_agent(
    model='gpt-4o',
    checkpointer=InMemorySaver(),
    tools=[search_web,send_email,delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_web":False,
                "send_email":True,
                "delete_record":True
            }
        )
    ]
)

confi = {"configurable":{"thread_id":"test-1"}}




In [5]:

result = agent.invoke({"messages":[{"role":"user","content":" send the email to @bigcompany.com  about Question 12 "}]},
confi)
result

{'messages': [HumanMessage(content=' send the email to @bigcompany.com  about Question 12 ', additional_kwargs={}, response_metadata={}, id='b65ee24d-e413-43e0-b8ec-a7c71d3029fd'),
  AIMessage(content='Could you please provide more details about the content of the email you\'d like me to send regarding "Question 12"? Specifically, the recipient\'s full email address, the subject line, and the content of the body would be helpful.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 111, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ELR1XpDo

In [6]:

##  for approve

approving = agent.invoke(Command(
    resume={
        "decisions":[{
            "type":"approve"
        }]
    }
),
confi
)
print(result["messages"][0].content)

 send the email to @bigcompany.com  about Question 12 


#### Reject 

In [3]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import  tool
from langgraph.checkpoint.memory import  InMemorySaver
from langgraph.types import Command

@tool
def search_web(query:str)->str:
    """ search the web for information """
    return f" seach the web by query :{query}"

@tool
def send_email(recipient:str,subject:str,body:str)->str:
    """send the email to recipeint """
    return f"send the Email to {recipient} with subject : {subject}"

@tool
def delete_record(table:str,condition:str)->str:
    """delete the record from the database"""
    return f"delete the record from the table {table} where condition : {condition}"


agent = create_agent(
    model='gpt-4o',
    checkpointer=InMemorySaver(),
    tools=[search_web,send_email,delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "search_web":False,
                "send_email":True,
                "delete_record":True
            }
        )
    ]
)

confi2 = {"configurable":{"thread_id":"test-2"}}




In [4]:

agent.invoke({"messages":[{"role":"user","content":"delete the record the store management where active = false"}]}
,
confi2
)


{'messages': [HumanMessage(content='delete the record the store management where active = false', additional_kwargs={}, response_metadata={}, id='cef24a61-8853-4f4e-b95a-0db11df6d0bd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 107, 'total_tokens': 128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ELR7PvVHkRtZ14QPoL78lwqAFgdqC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07b77-bab8-7fe1-81f4-7ad4ea962a3b-0', tool_calls=[{'name': 'delete_record', 'args': {'table': 'store_managemen

In [5]:

reject_result = agent.invoke(
    Command(
        resume = {
            "decisions":[
                {
                    "type":"reject",
                    "reason":"due to the unwanted input"
                }
            ]
        }
    ),
    confi2
)
print(reject_result["messages"][-1].content)


It seems that you canceled the request to delete the record. If you have any questions or need further assistance, please let me know!


### ⚙️ Section 5: Custom Guardrail — Before-Agent Hook (Input Filter)

Use before_agent() to validate or block requests before any LLM processing begins.

- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [17]:

from langchain.agents import  create_agent
from langchain.agents.middleware import AgentMiddleware,hook_config,AgentState
from langchain_core.tools import  tool
from langgraph.runtime import  Runtime
from typing import Any



class contentfiltermiddleWare(AgentMiddleware):
    """deministric gaurdrails : block the request which we use banned keyword.
    it is done before the llm, so it save the cost of llm"""

    def __init__(self,banned_keywords:list[list]):
        super().__init__()
        self.banned_keywords = [(kw.lower()) for kw in banned_keywords]
    
    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:

        if not state["messages"]:
            return None
        
        first_mss = state["messages"][0]

        if first_mss.type !="Human":
            return None

        contents = first_mss.lower()
        for keyword in banned_keywords:
            if keyword in contents:
                print("we find banned keyword : ",keyword)

                return{
                    "messages":[
                        {
                            "role":"assistant",
                            "content":"sorry you input invalid or wrong words "
                            "please try again"
                        }
                    ],
                    "jump_to":"end"
                }
        return None

@tool
def search(query:str)->str:
    """search the information : """
    return f"seaerch query : {query}"


agent = create_agent(
    model='gpt-4o',
    tools = [search],
    middleware = [
        contentfiltermiddleWare(
            {
                "banned_keywords":["hack","melware","apple","virus"]
            }
        )
    ]
)




In [5]:

respone = agent.invoke({
    "messages":[{"role":"user",
    "content":"what is the user of ai Only(3 point) "}]
})

print(respone["messages"][-1].content)

Here are three primary uses of AI:

1. **Automation**: AI is extensively used to automate repetitive and mundane tasks, which increases efficiency and productivity. This is valuable in various industries, including manufacturing, logistics, and customer service.

2. **Data Analysis**: AI is adept at analyzing vast volumes of data to uncover trends, patterns, and insights. This capability is beneficial in sectors such as healthcare for predictive analytics, finance for fraud detection, and marketing for consumer behavior analysis.

3. **Personalization**: AI powers personalized experiences by adjusting content, recommendations, and services based on individual user preferences and behaviors. This is commonly seen in recommendation engines for streaming services, e-commerce platforms, and digital advertising.


In [6]:

## using wrong messages

respone = agent.invoke({
    "messages":[{"role":"user",
    "content":"How i can hack the other software easily Only(3 point) "}]
})

print(respone["messages"][-1].content)

I’m sorry, I can’t assist with that.


#### 🔍 Section 6: Custom Guardrail — After-Agent Hook (Output Safety)
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [10]:

from typing import Any
from langchain.agents import  create_agent
from langchain.agents.middleware import AgentMiddleware,AgentState,Runtime,hook_config
from langchain_core.tools import  tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage


class safeguarddrailmode(AgentMiddleware):
    """
    guardrail base model : it give the respone after the request :
    but before the user reach
    """
    def __init__(self):
        # is used when you create a child class that inherits from a parent class.
        # AgentMiddleware → parent class . it is child class

        super().__init__()
        self.agent_model = ChatOpenAI(model='gpt-4o-mini',temperature=0)
    
    @hook_config(can_jump_to=["end"])
    def after_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] |None:
        # Contains the current state of the agent
        # Contains runtime information/context while the agent is running
        # The function can return a dictionary with string keys and any type of values
        if not state["messages"]:
            return None
        
        last_mss = state["messages"][-1]
        if not isinstance(last_mss,AIMessage):
            return None
        
        prompt = f"""
        Evaluate if this AI response is safe and appropriate for users.
        Respond with only 'SAFE' or 'UNSAFE'). 
        input{last_mss}
        """
        result = self.agent_model.invoke(prompt)

        if 'UNSAFE' in result.content.upper():
            print("give input is not suitable for to ask")
            last_mss.content = (
                "user request is bloked by system",
                "please try new one for to get good respone. please"
            )
        
        return None


@tool
def support(quer:str)->str:
    """for general purpose"""
    return f"output : {quer}"

agent = create_agent(
    model='gpt-4o',
    tools = [support],
    middleware = [safeguarddrailmode()]
)



In [18]:

#with safe question 

respone = agent.invoke({"messages":[{"role":"user","content":"what is the ai simple(2 line) ?"}]})

print(respone["messages"][-1].content)

AI, or Artificial Intelligence, is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, and learning from data.


In [19]:

## with unsafe question 

respone = agent.invoke({"messages":[{"role":"user","content":"how to hack the software ?"}]})

print(respone["messages"][-1].content)

I can't assist with that.



### 🧱 Section 7: Layered / Combined Guardrails
- User Input
-             ↓
- [Layer 1] ContentFilterMiddleware    ← Deterministic input filter
-             ↓
- [Layer 2] PIIMiddleware (input)      ← PII redaction on input
-              ↓
- [Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
-              ↓
- [Layer 4] PIIMiddleware (output)     ← PII redaction on output
-              ↓
- [Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
-              ↓
- User Response

In [2]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [14]:

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware,HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import  InMemorySaver



@tool 
def search(query:str)->str:
    """search the information : """
    return f"result : {query}"

@tool 
def send_to_email(recipient:str)->str:
    """send email"""
    return f"send to {recipient}"


good = create_agent(
    model='gpt-4o',
    checkpointer= InMemorySaver(),
    tools = [search,send_to_email],
    middleware = [

        # layer - 1 checking content 

        contentfiltermiddleWare(banned_keywords= ["hack","malware","virus"]),

        # layer -2  hiding import messages for input

        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_input = True,
            apply_to_output = True
        ),

        # layer - 3 checking for human approve 

        HumanInTheLoopMiddleware(
            interrupt_on = {
                "sent_to_email":True,
                "search":False
            }
        ),

        #  layer - 4 checking for again hiding import messages for output

        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_output = True
        ), 

        #  layer - 5 using safeguardrail 

        safeguarddrailmode()

    ]
)

confi = {"configurable":{"thread_id":"test_1.1"}}

In [15]:

good_respone = good.invoke({"messages":[{"role":"user","content":"send the email to @raiking123gmail.com about what is water (definition only (2 line))"}]}
,confi)


In [16]:
good_respone["messages"][-1].content

'The email has been sent to [REDACTED_EMAIL] with the definition of water.'


### 🏥 Section 8: Real-World Use Case — Healthcare Chatbot
- Blocks off-topic or harmful requests
- Redacts patient PII (emails, credit card numbers)
- Requires human approval before booking appointments
- Validates that outputs are medically appropriate

In [ ]:

from typing import  Any
from langchain.agents.middleware import  AgentMiddleware,AgentState,hook_config,PIIMiddleware,HumanInTheLoopMiddleware
from langchain.agents import  create_agent
from langchain_core.tools import  tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import  AIMessage
from langgraph.runtime import Runtime


class Healthcaresafty(AgentMiddleware):
    """block the harmful and non-medical thing for inputing :"""

    block_items = ["suicide method","weapon","murder","hack","drug","drug synthesis","self_harm"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:

        if not state["messages"]:
            return None
        
        first_mss = state["messages"][0]
        if first_mss.type != "Human":
            return None
        
        contents = first_mss.content.lower()
        for item in self.block_items:
            if item in contents:
                return {
                    "messages":[
                        {"role":"assistant",
                        "content":("This type of information is not valid or are not allowed to input soryy",
                        "please entry useful input",
                        "If you're in crisis, please call 100 or your local emergency number."
                        )
                        }
                    ],
                    "jump_to":"end"
                }
        return None



class MedicalOutputValidation(AgentMiddleware):
    """provide the output only valide"""

    addtional_info = """\n\n this is the general health information. Not for medical adive , if have that type question 
                     please visit the appropriate place"""

    
    @hook_config(can_jump_to=["end"])
    def after_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] |None:
        if not state["messages"]:
            return None
        
        last_mss = state["messages"][-1]

        if not isinstance(last_mss,AIMessage):
            return None
        

        if "medical advice" not in last_mss.content.lower():
            last_mss.content += self.additional_info
        

        return None


@tool 
def search_symptom(symptom:str)->str:
    """search the information about medical symptiom"""
    return f"search the information about : {symptom} : Please consult with doctor"


@tool
def book_appointment(patient_name:str,doctor:str,date:str)->str:
    """Appointment for checking health"""
    return f"{patient_name} is appointed in {date} by the doctor {doctor}"


@tool

def medical_info(medical:str):
    """This is the for the general medical information """
    return f"for medical information : {medical}. please visit near the hosiptal"


hosital_chat_bot = create_agent(
    model = 'gpt-4o',
    tools = [search_symptom,book_appointment,medical_info],
    checkpointer = InMemorySaver(),
    system_prompt = (
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis." ),
    
    middleware=[
        
        #  first layer - 1

        Healthcaresafty(),

        #  middle layer - 2 

        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = True,
        ),

        HumanInTheLoopMiddleware(
            interrupt_on = 
                {
                    "search_symptom":False,
                    "book_appointment":True,
                    "medical_info":False
                }
        ),

        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_output = False,
            apply_to_input = False
        ),

        #  last - layer

        MedicalOutputValidation()

    ]

)

        





In [4]:

confi = {"configurable":{"thread_id":"test_1"}}

question = hosital_chat_bot.invoke({"messages":[{"role":"user","content":" ram has appointed in july 3 by sita doctor"}]},
confi)
question


{'messages': [HumanMessage(content=' ram has appointed in july 3 by sita doctor', additional_kwargs={}, response_metadata={}, id='5c33b711-c6b1-4a03-a66f-d3f963ddfa07'),
  AIMessage(content="It seems like you want to book an appointment for Ram with Dr. Sita on July 3rd. I'll need to confirm a couple of details with you:\n\n- Ram's full name\n- Time preference for the appointment, if any\n\nPlease provide this information so I can assist you further.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 143, 'total_tokens': 204, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_3664e486bd', 'id

In [5]:

# test - 1

question["messages"][-1].content

"It seems like you want to book an appointment for Ram with Dr. Sita on July 3rd. I'll need to confirm a couple of details with you:\n\n- Ram's full name\n- Time preference for the appointment, if any\n\nPlease provide this information so I can assist you further."

In [9]:

## test - 2 
question = hosital_chat_bot.invoke({"messages":[{"role":"user","content":"How  can i used the drug synthesis at home"}]},
confi)
print(question["messages"][-1].content)

I'm sorry, but I cannot assist with information on synthesizing drugs at home. It's important to use medications that are prepared and prescribed by qualified professionals to ensure safety and effectiveness. If you have questions about legitimate medications, how to use them, and their effects, feel free to ask. Always consult a healthcare provider for any medical guidance you need.


In [14]:

## test 3 
confi = {"configurable":{"thread_id":"test_2"}}

question = hosital_chat_bot.invoke({"messages":[{"role":"user","content":"Book me the dr.Ram appointment on 23rd june"}]},
confi)

from langgraph.types import Command
approve = hosital_chat_bot.invoke(
    Command(resume={
        "descisions":[
            {
                "type":"approve"
            }
        ]
    }),
    confi
)
print(approve["messages"][-1].content)

I'll be happy to help you book an appointment. Could you please provide your full name for the booking?


In [21]:
## test - 4
question = hosital_chat_bot.invoke({"messages":[{"role":"user","content":"this is my personal account @sita@gmail.com please tell me about the headache"}]},
confi)
print(question["messages"][-1].content)

I found some information about headaches for you. Headaches can be caused by various factors including tension, stress, dehydration, and more serious underlying conditions. It's essential to pay attention to the nature and duration of the headache to determine if medical attention is needed.

Common types:
- **Tension headaches:** Often feel like a band around the head and are linked with stress.
- **Migraine headaches:** Characterized by throbbing pain, often accompanied by nausea and sensitivity to light or sound.
- **Cluster headaches:** Occur in patterns or clusters and can be extremely painful.

Please ensure to consult a healthcare professional for a proper diagnosis and treatment plan. If you'd like to book an appointment with a doctor or need more information on specific types of headaches, feel free to let me know!


In [23]:

# test -4

question = hosital_chat_bot.invoke({"messages":[{"role":"user","content":"How  can i used use of vegitable at home. Only two line"}]},
confi)
print(question["messages"][-1].content)

Incorporate vegetables into your meals by adding them to salads, stir-fries, or smoothies. They provide essential vitamins and minerals for a balanced diet.


### Project - ChatBot of school app

- only for student 
- only education relate material are avaiable 
- not for other information 


In [ ]:

import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware,AgentMiddleware,AgentState,hook_config,PIIMiddleware
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langgraph.runtime import  Runtime



class InputCheckBeforeLLm(AgentMiddleware):
    """check the input before the llm"""

    banned_kewords =["hack","virus","amdin password","melware","weapon"]

    @hook_config(can_jump_to=["end"])
    def beforeagent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:

        if not state["messages"]:
            return None
        
        first_mss = state["messages"][0]

        if not first_mss.type != "human":
            return None
        
        contents = first_mss.content.lower()
        for word in self.banned_kewords:
            if word in contents:
                return {
                    "messages":[{
                        "role":"assistant",
                        "content":("sorry this is not allowed for to input : "
                        "only valid input are allowed to input sorry : ")
                    }],
                    "jump_to":"end"
                }
        
        return None


class OutputCheckAfterLLm(AgentMiddleware):
    """check the output this is useful for user or not"""

    additional_info = "\n This chatbot only for education search not for other "
    

    @hook_config(can_jump_to=["end"])
    def afteragent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:

        if not state["messages"]:
            return None
        
        last_mss = state["messages"][-1]

        if not isinstance(last_mss,AIMessage):
            return None
        
        if "education" not in last_mss.content.lower():
            last_mss.content += self.additional_info
        
        return None



@tool
def school_info(query:str)->str:
    """search about the educational information like school ,colleges,place"""
    return f"result  {query}"

@tool
def other_info(quer:str)->str:
    """search the information which is not relate to the education"""
    return f"output : {quer} "

@tool
def school_material(query:str)->str:
    """search the requirement of the student education such as course,year to complete course,best course etc"""
    return f"output : {query}"


student_chat_bot = create_agent(
    model = 'gpt-4o',
    tools=[school_info,other_info,school_material],
    checkpointer = InMemorySaver(),
    system_prompt = ("are the helpful assistant"
                    "provide the education information only"
                    "Don't provide other information which is not relate to education "
                    "always give best adivise to the student ")
    ,
    middleware = [

        InputCheckBeforeLLm(),

        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = False,
            apply_to_output = True
        ),
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "school_info":False,         # run without human approve or interrupt
                "other_info":True,           # need human approve to run  
                "school_material":False       
            }
        ),
         PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_input = True,
            apply_to_output = True
         ),
        
        OutputCheckAfterLLm()
    ]
                
)


In [2]:

confi_1 = {"configurable":{"thread_id":"test_1"}}

testing = student_chat_bot.invoke({"messages":[{"role":"user","content":"what is Hulk"}]},
confi_1)


testing["messages"][-1].content

''

In [ ]:

### test - 2

testing = student_chat_bot.invoke({"messages":[{"role":"user","content":"show me the full course of BCA in nepal"}]},
confi_1)
testing["messages"][-1].content


"The Bachelor of Computer Applications (BCA) program in Nepal typically spans three years and offers a comprehensive curriculum designed to equip students with foundational and advanced knowledge in computer applications. Here's a general breakdown:\n\n**Year 1: Introduction and Basics**\n1. Fundamentals of Information Technology\n2. Digital Logic\n3. Mathematics I (Calculus and Algebra)\n4. Programming in C\n5. Probability and Statistics\n6. Technical Communication\n7. Introduction to Management\n\n**Year 2: Core Computing Concepts**\n1. Data Structures and Algorithms\n2. Software Engineering\n3. Web Technology and Programming\n4. Object-Oriented Programming with C++\n5. Mathematics II (Discrete Mathematics)\n6. Database Management Systems\n7. Accountancy and Financial Management\n\n**Year 3: Advanced Topics and Specialization**\n1. Operating Systems Concepts\n2. Computer Networks\n3. Multimedia Systems\n4. Java Programming\n5. Minor Project Work\n6. Internship/Industrial Attachment\n

In [9]:

### test - 3 

testing = student_chat_bot.invoke({"messages":[{"role":"user","content":"Show me the top colleges of world (ten)"}]},
confi_1)
print(testing["messages"][-1].content)


Here are ten of the world's top colleges, celebrated for their academic excellence, research initiatives, and global influence:

1. **Massachusetts Institute of Technology (MIT)** - USA
2. **Stanford University** - USA
3. **Harvard University** - USA
4. **California Institute of Technology (Caltech)** - USA
5. **University of Oxford** - United Kingdom
6. **University of Cambridge** - United Kingdom
7. **ETH Zurich - Swiss Federal Institute of Technology** - Switzerland
8. **University of Chicago** - USA
9. **Imperial College London** - United Kingdom
10. **University College London (UCL)** - United Kingdom

These colleges are recognized for their distinguished academic environments and significant contributions to research and innovation. Rankings can vary slightly based on specific evaluation criteria such as research quality, faculty credentials, and global impact.


In [10]:

### test - 4

testing = student_chat_bot.invoke({"messages":[{"role":"user","content":"why we eat meat?"}]},
confi_1)
print(testing["messages"][-1].content)


In [ ]:
### test - 5

testing = student_chat_bot.invoke({"messages":[{"role":"user","content":"i want to hack something"}]},
confi_1)

print(testing["messages"][-1].content)

I cannot assist with any illegal activities. If you are interested in ethical hacking and cybersecurity, I recommend pursuing legitimate educational courses and certifications. These programs teach you how to protect systems and networks. Remember, ethical hacking should always be conducted with permission and within the legal boundaries.



### Flight Chat Bot

- Only flight Information 
- Booking system
- send to email about flight delay
- No other information which is not relate to the fight Chat Bot


In [1]:

import os
from typing import Any
from langgraph.types import Command
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import  InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain.agents import  create_agent
from langchain.agents.middleware import AgentState,AgentMiddleware,HumanInTheLoopMiddleware,hook_config,PIIMiddleware

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


class InputCheckBeforeLLM_f(AgentMiddleware):
    """this is will check the input before reaching to the LLM"""

    banned_keywords = ["hack","withoutpayement","virus","melware","admin password","suicide"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:

        if not state["messages"]:
            return None
        
        first_mss = state["messages"][0]
        if first_mss.type != "Human":
            return None
        
            
        contents = first_mss.content.lower()
        for word in self.banned_keywords:
            if word in contents:
                return {
                    "messages":[{
                        "role":"assistant",
                        "content":("i can't help you sorry : "
                        "please try again with useful words or query"
                        "Don't keep this thing in your mind ")
                    }],
                    "jump_to":"end"
                }
        
        return None


class OutputCheckAfterLLm_f(AgentMiddleware):

    """check the output which is generate by the LLm"""

    addition_info = "this is only for the flight information not for other "

    @hook_config(can_jump_to=["end"])
    def after_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:

        if not state["messages"]:
            return None
        
        last_mss = state["messages"][-1]

        if not isinstance(last_mss,AIMessage):
            return None
            
        if "flight" not in last_mss.content.lower():
            last_mss.content += self.addition_info
        

        return None


@tool 
def flight_information(query:str)->str:
    """Search for flight information such as airlines, flight schedules, airports, and flights for a specified country or location"""
    return f"output : {query}"


@tool
def other_information(query:str)->str:
    """Search for general information that is not related to flights, airlines, aviation, or air travel."""
    return f"Output : {query}"

@tool
def flight_price(query:str)->str:
    """Search for flight ticket prices based on route, travel class, and other requirements"""
    return f" output : {query}"

@tool
def booking(name:str,date:str,time:str)->str:
    """About the booking the flight ticket"""

    return f"{name} has booked the ticket on {date} time {time}"

@tool
def send_email(recipient:str,subject:str,body:str)->str:
    """send email"""

    return f"Email has send to {recipient} with {subject} about {body}"



flight_chat_bot = create_agent(
    model='gpt-4o',
    tools = [flight_information,other_information,flight_price,booking,send_email],
    checkpointer = InMemorySaver(),
    system_prompt = ("You are a flight information assistant. "
    "IMPORTANT: Do not answer user questions using your own knowledge. "
    "You MUST use an appropriate tool to get information before giving an answer. "
    "Always use a tool when answering a user's request."),
    
    middleware = [

        InputCheckBeforeLLM_f(),

        PIIMiddleware(
            "email",
            strategy = "redact",
            # apply_to_input = True,
            apply_to_output= False
        ),
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "flight_information":False,     # not require approve
                "other_information":True,       # require approve
                "flight_price":False,
                "booking":True,
                "send_email":True
            }
        ),

        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_output = True,
        ),

        OutputCheckAfterLLm_f()

    ]
)




In [36]:

confi1 = {"configurable":{"thread_id":"test-1"}}

In [74]:
## test - 0 

## relate to the price of flight
chatting = flight_chat_bot.invoke({"messages":[{"role":"user","content":"What are the lowest and highest flight ticket prices of Nepal Airlines in Rs. for Economy Class and Business Class?"}]},
confi1)


print(chatting["messages"][-1].content)


Here are the flight ticket prices for Nepal Airlines:

- **Lowest Flight Ticket Price in Economy Class (in Rs.):** "output : lowest flight ticket price in Nepal Airlines Economy Class in Rs."
- **Highest Flight Ticket Price in Business Class (in Rs.):** "output : highest flight ticket price in Nepal Airlines Business Class in Rs."

If you need more information or assistance, feel free to let me know!


##### note - 
- Chatbot only work when you are all tools are correct because it check all tools for to run your code

In [39]:

### test - 1

chatting = flight_chat_bot.invoke({"messages":[{"role":"user","content":"How many country nepal get flight directly. show me 5 country?"}]},
confi1)


print(chatting["messages"][-1].content)


Nepal offers direct flights to various countries. Here are five countries that have direct flight connections with Nepal:

1. India
2. Qatar
3. United Arab Emirates
4. Thailand
5. Malaysia

For more detailed information, please consult the airlines or a flight information website.


In [49]:

### test - 2

# checking by asking not relate the flight question 
confi11 = {"configurable":{"thread_id":"test-3"}}

chatting = flight_chat_bot.invoke({"messages":[{"role":"user","content":"what do you mean by meat of pork?"}]},
confi11)
print(chatting["messages"][-1].content)



In [50]:
chatting

{'messages': [HumanMessage(content='what do you mean by meat of pork?', additional_kwargs={}, response_metadata={}, id='d653a006-d5a2-40a4-9dab-b09ab025aa8c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 227, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ENJb74MBCxyqWLNHnCXjq2AgSjiWa', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09627-c68a-7922-b933-b582ddd87d07-0', tool_calls=[{'name': 'other_information', 'args': {'query': 'meat of pork meaning'}, 'id': 'call_o

In [59]:

#test - 3

confi11 = {"configurable":{"thread_id":"booking-3"}}
chatting = flight_chat_bot.invoke({"messages":[{"role":"user","content":"Book the flight for me in july 23rd nepal to delhi time 1:30pm"}]},
confi11)
chatting



{'messages': [HumanMessage(content='Book the flight for me in july 23rd nepal to delhi time 1:30pm', additional_kwargs={}, response_metadata={}, id='29643ea3-c306-4b86-a7fc-c058db072401'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 239, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ENJjvD1WcvbuQkI9fcMOm4ktdFR31', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09630-1a8d-7f92-a14b-824880cedf86-0', tool_calls=[{'name': 'booking', 'args': {'name': 'Flight from Nepal t

In [60]:

# test - 3 for approve 
print("before not approve : ")
print(chatting)
print()
print("Successfully booked : ")
approve = flight_chat_bot.invoke(Command(
    resume = {
        "decisions":
            [
                {
                "type":"approve"
            }
            ]
        
    }
),
confi11
)

print(approve["messages"][-1].content)

before not approve : 
{'messages': [HumanMessage(content='Book the flight for me in july 23rd nepal to delhi time 1:30pm', additional_kwargs={}, response_metadata={}, id='29643ea3-c306-4b86-a7fc-c058db072401'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 239, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ENJjvD1WcvbuQkI9fcMOm4ktdFR31', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09630-1a8d-7f92-a14b-824880cedf86-0', tool_calls=[{'name': 'booking', 'args': {'name': 

In [69]:

## test - 4 
## relate to the email send 

confi1 = {"configurable":{"thread_id":"email-1"}}
chatting = flight_chat_bot.invoke({"messages":[{"role":"user","content":"send the email to ram@gmail.com on subject filght delay with body due to the bad weather "}]},
confi1)
chatting

{'messages': [AIMessage(content='Hello! How can I assist you with your flight-related queries today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 214, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ffd8308b42', 'id': 'chatcmpl-ENJwmaCC0HN9QgktGWpFrxiZZ179V', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0963c-43b9-7a02-926c-eaa10a7bab36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 214, 'output_tokens': 15, 'total_tokens': 229, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_

In [70]:

# with approve 

print("After Approved : ")
approved = flight_chat_bot.invoke(Command(
    resume = {
        "decisions":[{
            "type":"approve"
        }]
    }
),
confi1
)
print(approved["messages"][-1].content)

After Approved : 
The email has been sent to the specified address with the subject "flight delay" and the body "due to the bad weather." If there's anything else you need, feel free to ask!



### ChatBot of Information Technology 

- get information only about the information technology 
- not other information are avaialable 
- send email _ approve 

In [22]:

import os

from langgraph.types import  Command
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware,AgentMiddleware,AgentState,PIIMiddleware,hook_config
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

class InputCheckBeforeLLM_i(AgentMiddleware):

    banned_keywords = ["hack","virus","admin password","other password","melware","suicide"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:

        if not state["messages"]:
            return None
        
        first_mss = state["messages"][0]

        if first_mss.type !="Human":
            return None
        
        contents = first_mss.content.lower()
        for word in self.banned_keywords.lower():
            if word in contents:
                return {
                    "messages":[{
                        "role":"assistant",
                        "content":("This Chatbot is only for the information Technology : "
                        "please ask the question or query relate to the information Technology:")
                    }],
                    "jump_to":"end"
                }
        
        return None


class OutputcheckAfterLLM_t(AgentMiddleware):

    additional_info = "This Chatbot is only for the information Technology"

    it_keywords = [
    "python",
    "programming",
    "computer",
    "software",
    "hardware",
    "database",
    "network",
    "networking",
    "web development",
    "cybersecurity",
    "artificial intelligence",
    "machine learning",
    "information technology",
    "langchain",
    "coding"
]

    @hook_config(can_jump_to = ["end"])
    def after_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]| None:

        if not state["messages"]:
            return None
        
        last_mss = state["messages"][-1]

        if not isinstance(last_mss,AIMessage):
            return None
        
        if not any((kw in last_mss.content.lower())for kw in self.it_keywords):
            last_mss.content += self.additional_info
        

        return None

@tool
def information_technology(query:str)->str:
    """search the information which is related to information technology"""

    return f"output : {query}"

@tool
def other_information(query:str)->str:
    """search the information which is not relate to the information technology such as animal ,human beings,etc"""
    return f"outPut : {query}"

@tool 
def send_email(recipient:str,subject:str,body:str)->str:
    """send the email """
    return f"send email to {recipient} with subject {subject}  and with body {body}"


IT_chat_bot = create_agent(
    model = 'gpt-4o',
    tools = [information_technology,other_information,send_email],
    checkpointer = InMemorySaver(),
    system_prompt = (
        "your are the information technology chatBot."
        "IMPORTANT: Do not answer user questions using your own knowledge. "
        "You MUST use an appropriate tool to get information before giving an answer. "
        "Always use a tool when answering a user's request."
    ),

    middleware = [
        InputCheckBeforeLLM_i(),

        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = True
        ),

        HumanInTheLoopMiddleware(
            interrupt_on = {
                "information_technology":False,     #no approve
                "other_information":True,           #take approve
                "send_email":True
            }
        ),

        PIIMiddleware(
            "credit_card",
            strategy = "mask",
            apply_to_output = True
        ),

        OutputcheckAfterLLM_t()
    ]

)




In [2]:
confi = {"configurable":{"thread_id":"test_1"}}

In [6]:

#test - 1 

respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"what is IP4 ? "}]},
confi)
print(respon["messages"][-1].content)

IPv4 stands for Internet Protocol version 4. It is a protocol used for identifying devices on a network through an addressing system. IPv4 uses a 32-bit address scheme that allows for a total of 4.3 billion unique addresses, which is used to route traffic on the internet.This Chatbot is only for the information Technology


In [9]:

# test -2 

confi = {"configurable":{"thread_id":"test_2"}}
respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"what is heart attrack and veins ? "}]},
confi)

respon


{'messages': [HumanMessage(content='what is heart attrack and veins ? ', additional_kwargs={}, response_metadata={}, id='5b394df4-c270-469a-bf22-578cd58299df'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 161, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-ENYO9u5p0YHJrScUYlNOiW2TaUtOr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0998b-5641-7af2-b337-b6007084701b-0', tool_calls=[{'name': 'other_information', 'args': {'query': 'heart attack definition'}, 'id': 'ca

In [23]:

#test - 3

confi = {"configurable":{"thread_id":"test_3"}}
respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"what is living being ? "}]},
confi)

respon

{'messages': [HumanMessage(content='what is living being ? ', additional_kwargs={}, response_metadata={}, id='21b9f808-08c6-47ae-816d-c73c67326576'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 157, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-ENYhPudvPaNpvoUkuSalQbB56ckZU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0999d-8cdb-7e50-bdce-9ae55f0b832b-0', tool_calls=[{'name': 'other_information', 'args': {'query': 'living being'}, 'id': 'call_OZRX5xhylCs3ThNeXt0

In [14]:

## test - 4

confi = {"configurable":{"thread_id":"test_4"}}

respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"my credit card is 5555 5555 5555 4444 and i used it for to buy Software of google. "}]},
confi)

print(respon["messages"][-1].content)



I'm sorry, but I can't assist with issues related to credit cards or financial transactions. You should contact your credit card provider or Google's customer service for assistance with any queries or concerns regarding your purchase. 

If you need help with the software you purchased or have any technology-related questions, please let me know!This Chatbot is only for the information Technology


In [15]:

# test - 5

respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"How i create a virus for to disturb the software?"}]},
confi)

print(respon["messages"][-1].content)

I'm sorry, I can't assist with that request.This Chatbot is only for the information Technology


In [20]:

## test - 6

confi = {"configurable":{"thread_id":"test_5"}}

respon = IT_chat_bot.invoke({"messages":[{"role":"user","content":"send the email to Dipika@gmail.com subject 'Ip4' and body is definition of Ip4"}]},
confi)

respon

{'messages': [HumanMessage(content="send the email to [REDACTED_EMAIL] subject 'Ip4' and body is definition of Ip4", additional_kwargs={}, response_metadata={}, id='c8a6de64-9c3d-4e14-a313-1a4c88d8be8d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 174, 'total_tokens': 192, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_1812855600', 'id': 'chatcmpl-ENYbex2KtEUw4GqNRNbbTR83okej7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09998-1c39-75c3-9ff4-bf49fdcd1f15-0', tool_calls=[{'name': 'information_technology', 'args'

In [21]:

print("after approved : ")

approve = IT_chat_bot.invoke(
    Command(
        resume={
            "decisions":[{
                "type":"approve"
            }]
        }
    ),
    confi
)

print(approve["messages"][-1].content)

after approved : 
The email with the subject "Ip4" and the definition of IPv4 has been sent successfully to the requested email address.This Chatbot is only for the information Technology
